In [0]:
%run ./config

In [0]:
fact_df=spark.read.format('delta').load(gold+'/fact_table')
fact_df.createOrReplaceTempView('fact_df')
loan_status_events_df=spark.read.csv(silver+'/loan_status_events.csv', header=True, inferSchema=True)

In [0]:
total_active_loans=spark.sql("""
                             select count(*) total_active_loans from fact_df where outstanding_amount>0
                             """)
total_active_loans.write.format('delta').mode('overwrite').save(gold+'/total_active_loans')

In [0]:
total_outstanding_amount=spark.sql("""
                             select round(sum(outstanding_amount),2) total_outstanding_amount from fact_df where outstanding_amount>0""")

total_outstanding_amount.write.mode('overwrite').format('delta').save(gold+'/total_outstanding_amount')

In [0]:
avg_urs_score=spark.sql("""
                             select round(avg(urs_score),2) avg_urs_score from fact_df 
                        """)
avg_urs_score.write.mode('overwrite').format('delta').save(gold+'/avg_urs_score')

In [0]:
repayments_df=spark.read.format('delta').load(gold+'/repayment')
repayments_df.createOrReplaceTempView('repayments_df')

In [0]:
percentage_overdue_loans=spark.sql("""
                                   with cte1 as (
                                       select  c1.loan_id, repayment_status
                                       from fact_df c1 join repayments_df c2 on c1.loan_id=c2.loan_id
                                       where outstanding_amount>0
                                   )
                                   select round((sum(case when repayment_status ='missed' then 1 else 0 end))*100/count(*),2) percentage_overdue_loans from cte1
                                   """)
percentage_overdue_loans.write.format('delta').mode('overwrite').save(gold+'/percentage_overdue_loans')

In [0]:
percentage_severely_overdue=spark.sql("""
                                   with cte1 as (
                                       select loan_id, repayment_status, days_past_due
                                       from repayments_df
                                   )
                                   select round(sum(case when repayment_status ='missed' and days_past_due>60 then 1 else 0 end)/count(*)*100,2) percentage_severely_overdue
                                   from cte1 
                                   """)
percentage_severely_overdue.write.format('delta').mode('overwrite').save(gold+'/percentage_severely_overdue')

In [0]:
df=spark.read.format('delta').load(gold+'/percentage_severely_overdue')
df.display()